# Security lifecycle: mandated support floors versus stated vendor support periods

**Claim cluster:** `C064`, `C065`, `C066`, `C069`, `C103`, `C105`, `C067`, `C068`  
**Source of truth:** `data/smarthome.duckdb` (read-only), via `src/viz_core.py`

In [ ]:
import os
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, clear_output, display

# Find the repo root by walking up until schemas/stage2.sql appears, so the
# notebook runs from notebooks/, from the repo root, or from anywhere else.
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "schemas" / "stage2.sql").exists():
        ROOT = _candidate
        break
else:  # pragma: no cover - only reachable outside a checkout
    raise FileNotFoundError("schemas/stage2.sql not found above " + str(_here))

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

# Option A topology: this wave reads one dedicated Stage 2 database. Point
# viz_core at it before import -- it resolves the path once at import time --
# so every fetch in this kernel uses the same file.
os.environ["SMARTHOME_DB"] = str(ROOT / "data" / "smarthome.duckdb")

import viz_core as viz  # noqa: E402

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

# Renderers matter for two reasons. "browser" (plotly's default) opens a
# window when a figure is shown outside a live kernel, which breaks headless
# execution. The bare "notebook" renderer in plotly 7 emits only text/html,
# carrying no figure payload, so a notebook could run clean and still show
# nothing. "plotly_mimetype+notebook" emits the figure JSON for JupyterLab and
# an inline HTML fallback elsewhere -- which is also what the headless test
# asserts on.
pio.renderers.default = "plotly_mimetype+notebook"

# Every figure in this notebook reads only from data/smarthome.duckdb (read-only).
CLAIM_IDS = ('C064', 'C065', 'C066', 'C069', 'C103', 'C105', 'C067', 'C068')
assert viz.DEFAULT_DB_PATH.endswith("smarthome.duckdb"), viz.DEFAULT_DB_PATH

claims = viz.fetch_claims(list(CLAIM_IDS))
sources = viz.sources_for_claims(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
cluster_metrics = (
    viz.claim_metrics_lookup(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
)

# Corpus-wide row counts, read once so a notebook can state the size of what it
# is drawing from without confusing a whole-corpus count with the size of its
# own (possibly empty) claim cluster.
CORPUS = viz.connect_db()
try:
    corpus_counts = {
        table: CORPUS.execute(f"SELECT COUNT(1) FROM {table}").fetchone()[0]
        for table in ("stage2_claim", "metric", "entity", "source_mirror",
                      "predicate", "extraction_decision")
    }
finally:
    CORPUS.close()

def provenance(claim_id):
    """Collapsible evidence panel for one claim in this cluster."""
    panel = viz.render_provenance_panel(claim_id)
    return widgets.VBox([panel])

def show(*items):
    """Display data frames, HTML, or figures, one per argument."""
    for item in items:
        display(item)

def confidence_legend():
    """Show the colour coding applied to confidence across all figures."""
    display(HTML(
        "<b>Confidence colour coding:</b> "
        f"<span style='color:{viz.CONFIDENCE_COLORS['high']}'>high</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['medium']}'>medium</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['low']}'>low</span>"
    ))

## Provenance

Support periods are the most quotable and most easily misread numbers in the corpus: a mandated floor and a vendor's stated period are different kinds of number. The provenance panels below separate the regulation claims from the vendor-report claims before they are drawn on one axis.

In [ ]:
for claim_id in CLAIM_IDS:
    display(HTML(f"<h4>{claim_id} &mdash; {claims.set_index('local_id').loc[claim_id, 'claim_text']}</h4>"))
    display(provenance(claim_id))
confidence_legend()

## Primary evidence

### Stated support periods by device category, against the CRA floor

The dotted line is the CRA's five-year minimum support floor (`M053`, `documented fact`, high confidence). Bars are the periods the corpus records per device category, drawn as ranges where the source stated a range. Where a period is at or below the floor, the hover still shows the source, because the interesting cases are the ones where the floor is not yet the binding constraint.

In [ ]:
SUPPORT_SECTION = "Support Periods and Update Mechanisms by Device Category"
support = viz.section_metrics(SUPPORT_SECTION)
show(support[["local_id", "metric_name", "value", "unit", "confidence",
               "source_ref", "conditions_stated"]].rename(
    columns={"local_id": "metric", "metric_name": "name",
             "source_ref": "source", "conditions_stated": "cond_stated"}))

# Only rows whose unit is years belong on a support-period axis. M063 (AI
# assistant data retention, 18 / 3 or 18 months) shares the section but is a
# different quantity, and M059's "3, 4, 5" is three separate periods for three
# hubs rather than a range, so it is expanded below instead of collapsed.
period_rows = []
for record in support.to_dict("records"):
    if record["unit"] != "years":
        continue
    values = viz.parse_numbers(record["value"])
    if not values:
        continue
    low = min(values)
    high = max(values)
    period_rows.append({
        "metric": record["local_id"],
        "name": record["metric_name"],
        "value": record["value"],
        "low": low,
        "high": high,
        # "3, 4, 5" is a list of three vendor periods, so the bar is a span of
        # the observed set rather than an author-stated range.
        "bounds_meaning": "stated range" if len(values) == 2 else "observed spread",
        "confidence": record["confidence"],
        "source": record["source_ref"],
    })
periods = pd.DataFrame(period_rows).sort_values("low")
show(periods)

floor_row = viz.metrics_by_id(["M053"])
floor_years = viz.numeric_value(floor_row["value"].iloc[0]) if not floor_row.empty else None

_records = periods.to_dict("records")
fig = go.Figure(go.Bar(
    x=[(r["low"] + r["high"]) / 2 for r in _records],
    y=list(reversed(periods["metric"] + " " + periods["name"])),
    orientation="h",
    marker=dict(color=[viz.CONFIDENCE_COLORS.get(c, viz.CONFIDENCE_COLORS[None])
                       for c in periods["confidence"]]),
    error_x=dict(
        type="data",
        array=[(r["high"] - r["low"]) / 2 for r in _records],
        arrayminus=[(r["high"] - r["low"]) / 2 for r in _records],
        visible=True,
    ),
    customdata=list(zip(periods["value"], periods["source"], periods["confidence"],
                        periods["bounds_meaning"])),
    hovertemplate=("%{y}<br>stated: %{customdata[0]} years"
                   "<br>source: %{customdata[1]}<br>confidence: %{customdata[2]}"
                   "<br>bars show the %{customdata[3]}<extra></extra>"),
))
if floor_years is not None:
    fig.add_vline(x=floor_years, line_dash="dot", line_color="#333333",
                  annotation_text=f"CRA minimum support floor: {floor_years:g} years (M053)",
                  annotation_position="top left")
fig.update_layout(
    title="Stated support periods by device category, against the CRA floor",
    xaxis_title="years of support", height=380, yaxis=dict(automargin=True),
)
fig.show()

## Supporting context

### Compliance is not the same as safety, and the corpus says so

`C064` and `C065` record the CRA's five-year floor together with the qualification that the floor is explicitly not a default: manufacturers can justify shorter. `C066` records that the US has no comparable federal lifecycle mandate and that `M070` records the UK PSTI as mandating no minimum support period at all. `C103` records that vendors state automatic updates while offering shorter windows, and `C105` records Aqara CVEs as evidence that documented features can be absent in practice.

The chart puts the regulatory instruments on one axis so the gap between *mandated*, *stated* and *observed* is legible in one glance.

In [ ]:
# Mandated instruments only: the CRA block plus the two rows that record the
# absence of a mandate (M070 UK PSTI, and the US comparison in C066).
MANDATE_IDS = ["M053", "M054", "M055", "M056", "M062", "M069", "M070"]
instruments = viz.metrics_by_id(MANDATE_IDS)
show(instruments[["local_id", "metric_name", "value", "unit", "confidence",
                  "source_ref", "section"]].rename(
    columns={"local_id": "metric", "metric_name": "name",
             "source_ref": "source", "confidence": "conf"}))

# Only the rows measured in years belong on a years axis. M055 is a date and
# M056 is a reporting deadline in hours; plotting them as years would be wrong,
# so they are shown in the table above and excluded from the figure.
_years_only = instruments[instruments["unit"] == "years"].copy()
_years_only["years"] = [viz.numeric_value(v) for v in _years_only["value"]]
_years_only = _years_only.dropna(subset=["years"])

fig = go.Figure(go.Bar(
    x=list(_years_only["years"]),
    y=list(_years_only["local_id"] + " " + _years_only["metric_name"]),
    orientation="h",
    marker=dict(color=[viz.CONFIDENCE_COLORS.get(c, viz.CONFIDENCE_COLORS[None])
                       for c in _years_only["confidence"]],
                pattern=dict(shape="", solidity=0.2)),
    customdata=list(zip(_years_only["value"], _years_only["confidence"],
                        _years_only["section"])),
    hovertemplate=("%{y}<br>stated: %{customdata[0]}"
                   "<br>confidence: %{customdata[1]}<br>section: %{customdata[2]}"
                   "<extra></extra>"),
))
fig.add_vline(x=floor_years or 5, line_dash="dot", line_color="#333333",
              annotation_text="CRA minimum support floor (M053)",
              annotation_position="top left")
fig.update_layout(
    title=("Lifecycle instruments stated in years. M070 records that UK PSTI "
           "mandates no minimum period, so it is a table row, not a bar."),
    xaxis_title="years", height=300, yaxis=dict(automargin=True),
)
fig.show()

display(HTML(
    "<p><b>M070</b> (UK PSTI minimum support period) is recorded as "
    "<code>none</code>. It cannot be drawn as a zero-height bar without "
    "implying a measured zero, so it is left as text: the UK regime mandates "
    "no floor at all, which is a different statement from mandating a short "
    "one. <b>M055</b> (December 2027) and <b>M056</b> (24 hours) are excluded "
    "from the years axis for the same reason.</p>"
))

## Uncertainty and gaps

### Why this page stops short of a security scorecard radar

The corpus records the scorecard's *shape* (`M064`-`M068`: 14 lifecycle controls, weights 1.0 and 0.7, a passing composite of 2.00 out of 3.00) but stores no per-control scores. A radar of 14 controls would therefore be drawn from the rubric rather than from results, which would misrepresent a scoring method as a measurement. The figures above are limited to numbers the database actually contains.

- **Support periods are vendor-reported ranges, not audited measurements.** `M057`, `M059` and `M060` carry medium confidence for that reason.
- **The CRA floor is prospective.** `M055` dates the major-obligation application to December 2027, so the floor is not yet the operative constraint for any device on the chart.
- **Camera support is a single recorded value.** `M058` gives 5 years for one vendor line; the corpus holds no per-vendor camera support table, so the 'none for Apple' reading from the plan is not drawn here as data.
- **`C068` is an inference**, not a measurement: the claim that households rarely VLAN-segment IoT devices is a judgement about behaviour with no survey behind it in this corpus.

In [ ]:
display(HTML("<h4>Claims in this cluster with no falsifier</h4>"
             "<p>A missing falsifier means the corpus recorded no observation "
             "that would disconfirm the claim. The panel above flags these "
             "claims in <code>no_falsifier</code>.</p>"))
show(claims.loc[claims["falsifier_missing"], ["local_id", "claim_type",
                                              "confidence"]]
     .rename(columns={"local_id": "claim", "claim_type": "type",
                      "confidence": "conf"}))

display(HTML("<h4>Metrics with conditions not stated in the source</h4>"))
_gaps = cluster_metrics.loc[~cluster_metrics["conditions_stated"].fillna(False)]
if _gaps.empty:
    display(HTML("<p>No unstated conditions among the cluster's metrics.</p>"))
else:
    show(_gaps[["local_id", "metric_name", "value", "unit"]].rename(
        columns={"local_id": "metric", "metric_name": "name"}))

display(HTML("<h4>Extraction decisions that changed or qualified this material</h4>"))
_decisions = pd.DataFrame(
    viz.connect_db().execute(
        "SELECT local_id, decision_type, description, resolution "
        "FROM extraction_decision "
        "WHERE decision_type IN ('evidence_downgrade', 'classification_conflict', "
        "                           'falsifier_absent', 'condition_absent', "
        "                           'boundary_absent', 'scope_boundary', "
        "                           'omission') "
        "ORDER BY local_id"
    ).fetchall(),
    columns=["decision", "type", "description", "resolution"],
)
show(_decisions)

## Reproducibility

Section-scoped metrics come from `viz.section_metrics` (exact section match, ordered by local id) and the regulatory comparison from an explicit metric-id list. Both are read-only queries against `data/smarthome.duckdb`.

In [ ]:
# The exact queries behind every figure above, re-run verbatim.
import duckdb
con = duckdb.connect(str(ROOT / 'data' / 'smarthome.duckdb'), read_only=True)
print('database:', con.execute('SELECT current_database()').fetchone()[0])
for table in ('stage2_claim', 'metric', 'entity', 'source_mirror',
               'predicate', 'workflow_stage', 'extraction_decision'):
    total = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table:22} {total:>6} rows')
con.close()